# 🏗️ BIM-RAG: Llama 3 on Colab
**Steps:** Install → Load model → Serve vLLM → Expose via ngrok → Optional fine-tune
> 📌 Free T4 GPU via Google Colab (15GB VRAM — enough for Llama 3 8B in 4-bit)

In [1]:
# ── Step 1: Install dependencies ─────────────────────────────────────
!pip install -q unsloth vllm pyngrok transformers trl peft datasets bitsandbytes accelerate
!pip install -q huggingface_hub
print('✅ Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/

In [2]:
# ── Step 2: Mount Google Drive (for saving/loading adapters) ──────────
from google.colab import drive
drive.mount('/content/drive')
ADAPTER_PATH = '/content/drive/MyDrive/bim_rag_lora'  # Where LoRA adapters are saved/loaded
DATASET_PATH = '/content/drive/MyDrive/finetuning_dataset.jsonl'  # Upload your .jsonl here
print(f'Drive mounted. Adapter path: {ADAPTER_PATH}')

Mounted at /content/drive
Drive mounted. Adapter path: /content/drive/MyDrive/bim_rag_lora


In [3]:
# ── Step 3: Load base model with Unsloth (4-bit QLoRA) ───────────────
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096
BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B-Instruct'
import os

# Load existing LoRA adapter if present, else base model
if os.path.exists(ADAPTER_PATH):
    print(f'Loading fine-tuned adapter from {ADAPTER_PATH}...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_PATH,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
else:
    print(f'Loading base model {BASE_MODEL}...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )

FastLanguageModel.for_inference(model)  # optimise for inference
print('✅ Model loaded')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model unsloth/Meta-Llama-3.1-8B-Instruct...
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.19.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

✅ Model loaded


In [ ]:
# ── Step 4: Start vLLM OpenAI-compatible server ─────────────────────
# NOTE: vLLM runs separately. This cell saves the model in HF format first,
# then launches vLLM pointing to it.
import subprocess, time, os

SERVE_MODEL_DIR = '/content/serve_model'
if not os.path.exists(SERVE_MODEL_DIR):
    print('Saving model for vLLM serving...')
    model.save_pretrained(SERVE_MODEL_DIR)
    tokenizer.save_pretrained(SERVE_MODEL_DIR)
    print('Model saved.')

# Launch vLLM in background
vllm_proc = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', SERVE_MODEL_DIR,
    '--host', '0.0.0.0',
    '--port', '8000',
    '--max-model-len', '4096',
    '--dtype', 'float16',
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('Starting vLLM server... (waiting 30s)')
time.sleep(30)
print('✅ vLLM server started on localhost:8000')

Saving model for vLLM serving...


In [ ]:
# ── Step 5: Expose vLLM via ngrok ────────────────────────────────────
from pyngrok import ngrok, conf

# Get a free ngrok auth token from https://dashboard.ngrok.com/
NGROK_AUTH_TOKEN = '3B3LcPHHXYtY6i23ww9Z5QctEPo_5bKLWh287wAKmR9yH6uFV'  # ← paste your token
conf.get_default().auth_token = NGROK_AUTH_TOKEN

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print(f'\n🚀 vLLM Public URL: {public_url}')
print('\n📋 Set this in your backend .env file:')
print(f'CUSTOM_LLM_URL={public_url}/v1')
print('LLM_MODEL_NAME=bim-llama')

In [ ]:
# ── Step 6: (Optional) Fine-tune on feedback data ────────────────────
# Upload data/finetuning_dataset.jsonl to your Google Drive first.
# Then run the export from your backend:
#   from src.finetuning.db_logger import FeedbackLogger
#   FeedbackLogger().export_finetuning_dataset('data/finetuning_dataset.jsonl')

import os
if not os.path.exists(DATASET_PATH) or os.path.getsize(DATASET_PATH) == 0:
    print('⚠️ No dataset found at', DATASET_PATH)
    print('Upload your finetuning_dataset.jsonl to Google Drive first.')
else:
    from datasets import load_dataset
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from unsloth import FastLanguageModel

    # Re-apply PEFT for training
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=SERVE_MODEL_DIR,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16,
        lora_dropout=0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )

    dataset = load_dataset('json', data_files=DATASET_PATH, split='train')

    def format_msgs(examples):
        texts = []
        for msgs in examples['messages']:
            t = ''
            for m in msgs:
                t += f"<|{m['role']}|>\n{m['content']}\n"
            texts.append(t)
        return {'text': texts}

    dataset = dataset.map(format_msgs, batched=True)

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_train_epochs=1,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=5,
            optim='adamw_8bit',
            output_dir='/content/train_output',
            seed=42,
        ),
    )

    print('🏋️ Fine-tuning started...')
    trainer.train()

    print(f'💾 Saving new adapter to {ADAPTER_PATH}...')
    model.save_pretrained(ADAPTER_PATH)
    tokenizer.save_pretrained(ADAPTER_PATH)
    open(DATASET_PATH, 'w').close()  # clear processed dataset
    print('✅ Fine-tuning complete! New adapter saved to Drive.')

In [ ]:
# ── Keep Colab alive (run this in a loop or use a keep-alive trick) ──
# The session will stay active while this cell runs
import time
print('🟢 Colab session is ACTIVE. vLLM server running.')
print('Set CUSTOM_LLM_URL in your .env and restart your RAG backend.')
print('Press Ctrl+C or interrupt this cell to stop.')
try:
    while True:
        time.sleep(60)
        print('.', end='', flush=True)
except KeyboardInterrupt:
    print('\nSession ended.')